In [ ]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, median_absolute_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone
from scipy.stats import randint, uniform
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import glob, os
import time
import sys
sys.path.append('..')
from src.preprocessing import FeatureExtractor
import mlflow
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
mlflow.set_experiment("kufar-notebook")

def log_model(model, X_train, y_train, X_test, y_test, run_name="run", log=False):
    
    if hasattr(model, "best_estimator_"):
        params = model.best_params_
        model = model.best_estimator_
    else:
        params = model.get_params()

    train_pred = model.predict(X_train)
    
    if log:
        y_pred = np.exp(model.predict(X_test))
    else:
        y_pred = model.predict(X_test)
        
    errors = y_test - y_pred

    metrics = {
        "train_mae": mean_absolute_error(y_train, train_pred),
        "train_r2": r2_score(y_train, train_pred),
        "test_mae": mean_absolute_error(y_test, y_pred),
        "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "test_r2": r2_score(y_test, y_pred),
        "test_mape": mean_absolute_percentage_error(y_test, y_pred),
        "test_median_ae": median_absolute_error(y_test, y_pred),
        "error_p95": np.percentile(np.abs(errors), 95),
    }

    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, "model")

### Loading the latest snapshot

In [ ]:
folder_path = r'..\data\raw'
file_type = '/*csv'

files = glob.glob(folder_path + file_type)

latest_file = max(files, key=os.path.getctime)

time = latest_file[12:].split(sep="_")

access_time = pd.Timestamp(
    year=int(time[0][0:4]),
    month=int(time[0][4:6]),
    day=int(time[0][6:8]),
    hour=int(time[1][0:2]),
    minute=int(time[1][2:4]),
    tz='Europe/Moscow'
)

df = pd.read_csv(latest_file)
df =  df[df["price_byn"] > 30]

### Train/test split

In [ ]:
y = df["price_byn"]
X = df.drop(["price_byn"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42)

### Training & evaluation helper

In [ ]:
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)

# Results table
results = []


def evaluate_model(name, pipeline, X_train, y_train, X_test, y_test, log=False):

    if log:
        pipeline.fit(X_train, np.log(y_train))
        y_pred = np.exp(pipeline.predict(X_test))
    else:
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    mape = mean_absolute_percentage_error(y_test, y_pred)

    cv = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring={
            "r2": "r2",
            "rmse": "neg_root_mean_squared_error",
            "mae": "neg_mean_absolute_error"
        },
        n_jobs=-1,
        return_train_score=False
    )

    segment_1 = y_test < 500
    segment_2 = (y_test >= 500) & (y_test < 3000)
    segment_3 = y_test >= 3000
    
    results.append({
        "Model": name,
    
        "Test R2": r2,
        "Test RMSE": rmse,
        "Test MAE": mae,
        "Test MedAE": median_absolute_error(y_test, y_pred),
        "Test MAPE": mape,
    
        "MAE <500": mean_absolute_error(
            y_test[segment_1],
            y_pred[segment_1]
        ),
    
        "MAE 500-3000": mean_absolute_error(
            y_test[segment_2],
            y_pred[segment_2]
        ),
    
        "MAE >3000": mean_absolute_error(
            y_test[segment_3],
            y_pred[segment_3]
        ),
    
        "CV R2": cv["test_r2"].mean(),
        "CV RMSE": -cv["test_rmse"].mean(),
        "CV MAE": -cv["test_mae"].mean(),
    })

def show_results():

    df = pd.DataFrame(results)

    if df.empty:
        print("No results")
        return

    numeric_cols = df.select_dtypes(include=np.number).columns

    # Metrics where higher is better
    max_metrics = [
        col for col in [
            "Test R2",
            "CV R2"
        ] if col in df.columns
    ]

    # Metrics where lower is better
    min_metrics = [
        col for col in [
            "Test RMSE",
            "Test MAE",
            "Test MedAE",
            "Test MAPE",

            "MAE <500",
            "MAE 500-3000",
            "MAE >3000",

            "CV RMSE",
            "CV MAE"
        ] if col in df.columns
    ]

    styled = (
        df.style
        .format("{:.4f}", subset=numeric_cols)
    )

    if max_metrics:
        styled = styled.highlight_max(
            subset=max_metrics,
            color="lightgreen"
        )

    if min_metrics:
        styled = styled.highlight_min(
            subset=min_metrics,
            color="lightgreen"
        )

    return styled

In [ ]:
categorical_features = [
    "brand",
    "processor",
    "rom_type",
    "os", 
    "videocard",
    "videocard_brand",
    "region",
    "matrix_type",
    "display_resolution",
    "ram_type"
]

cb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time)),
    ('regressor', CatBoostRegressor(
        iterations=1000,
        learning_rate=0.03,
        depth=7,
        loss_function='MAE',
        early_stopping_rounds=50,
        random_seed=42,
        logging_level='Silent',
        cat_features=categorical_features
    ))
])

xgb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time, GBM_XGB=True)),
    ('regressor', XGBRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=7,
        objective='reg:absoluteerror',
        random_state=42,
        enable_categorical=True
    ))
])


lgb_pipeline = Pipeline([
    ('extractor', FeatureExtractor(access_time, GBM_XGB=True)),
    ('regressor', LGBMRegressor(
        n_estimators=1000,
        learning_rate=0.03,
        max_depth=7,
        objective='mae', 
        random_state=42,
        verbose=-1
    ))
])

evaluate_model(
    "XGB",
    xgb_pipeline,
    X_train, y_train,
    X_test, y_test
)

evaluate_model(
    "CatBoost",
    cb_pipeline,
    X_train, y_train,
    X_test, y_test
)

evaluate_model(
    "LGB",
    lgb_pipeline,
    X_train, y_train,
    X_test, y_test
)

In [ ]:
show_results()

### Model performance on the high-price segment

The baseline struggles on expensive listings — large absolute errors dominate the loss because of the heavy right tail.

In [ ]:
y_pred = cb_pipeline.predict(X_test)
dif_column = abs(y_test-y_pred)
dif_df = pd.DataFrame({"test":y_test, "pred":y_pred, "dif":dif_column})
dif_df.sort_values("dif", ascending=False).head(30)

In [ ]:
show_results()

In [ ]:
catboost_param_dist = {
    'regressor__iterations':           randint(400, 1200),
    'regressor__learning_rate':        uniform(0.01, 0.09),
    'regressor__depth':                randint(4, 10),
    'regressor__l2_leaf_reg':          uniform(1, 9),
    'regressor__subsample':            uniform(0.6, 0.4),
    'regressor__colsample_bylevel':    uniform(0.6, 0.4),
    'regressor__min_data_in_leaf':     randint(1, 50),
}

xgb_param_dist = {
    'regressor__n_estimators':         randint(400, 1200),
    'regressor__learning_rate':        uniform(0.01, 0.09),
    'regressor__max_depth':            randint(3, 10),
    'regressor__subsample':            uniform(0.6, 0.4),
    'regressor__colsample_bytree':     uniform(0.6, 0.4),
    'regressor__reg_alpha':            uniform(0, 5),
    'regressor__reg_lambda':           uniform(1, 9),
    'regressor__min_child_weight':     randint(1, 20),
}

lgb_param_dist = {
    'regressor__n_estimators':         randint(400, 1200),
    'regressor__learning_rate':        uniform(0.01, 0.09),
    'regressor__max_depth':            randint(3, 10),
    'regressor__num_leaves':           randint(20, 150),
    'regressor__subsample':            uniform(0.6, 0.4),
    'regressor__colsample_bytree':     uniform(0.6, 0.4),
    'regressor__reg_alpha':            uniform(0, 5),
    'regressor__reg_lambda':           uniform(1, 9),
    'regressor__min_child_samples':    randint(5, 50),
}

cat_search = RandomizedSearchCV(
    clone(cb_pipeline), catboost_param_dist,
    n_iter=10, scoring='neg_mean_absolute_error',
    cv=5, n_jobs=-1, random_state=42
)
cat_search.fit(X_train, y_train)

xgb_search = RandomizedSearchCV(
    clone(xgb_pipeline), xgb_param_dist,
    n_iter=10, scoring='neg_mean_absolute_error',
    cv=5, n_jobs=-1, random_state=42
)
xgb_search.fit(X_train, y_train)

lgb_search = RandomizedSearchCV(
    clone(lgb_pipeline), lgb_param_dist,
    n_iter=10, scoring='neg_mean_absolute_error',
    cv=5, n_jobs=-1, random_state=42
)
lgb_search.fit(X_train, y_train)

evaluate_model("CatBoost_tuned", cat_search.best_estimator_, X_train, y_train, X_test, y_test)
evaluate_model("XGB_tuned",      xgb_search.best_estimator_, X_train, y_train, X_test, y_test)
evaluate_model("LGB_tuned",      lgb_search.best_estimator_, X_train, y_train, X_test, y_test)

In [ ]:
show_results()

In [ ]:
print("catboost\n", cat_search.best_params_)
print("xgb\n", xgb_search.best_params_)
print("lgbm\n", lgb_search.best_params_)

### Log-transforming the target

Prices follow a log-normal distribution, so we train on `log(y_train)` and invert with `exp(...)` at inference.

In [ ]:
# LightGBM
log_lgb_search = RandomizedSearchCV(
    clone(lgb_pipeline),
    lgb_param_dist,
    n_iter=30,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1,
    random_state=42
)

log_lgb_search.fit(X_train, log_y_train)


# XGBoost
log_xgb_search = RandomizedSearchCV(
    clone(xgb_pipeline),
    xgb_param_dist,
    n_iter=30,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1,
    random_state=42
)

log_xgb_search.fit(X_train, log_y_train)


# CatBoost
log_cat_search = RandomizedSearchCV(
    clone(cat_pipeline),
    cat_param_dist,
    n_iter=30,
    scoring='neg_mean_absolute_error',
    cv=5,
    n_jobs=-1,
    random_state=42
)

log_cat_search.fit(X_train, log_y_train)
log_xgb_search.fit(X_train, log_y_train)
log_lgb_search.fit(X_train, log_y_train)

In [ ]:
log_model(log_cat_search, X_train, log_y_train, X_test, y_test, run_name="log_cat_search27.05.2026", log=True)
log_model(log_xgb_search, X_train, log_y_train, X_test, y_test, run_name="log_xgb_search27.05.2026", log=True)
log_model(log_lgb_search, X_train, log_y_train, X_test, y_test, run_name="log_lgb_search27.05.2026", log=True)

evaluate_model("log_CatBoost_tuned", log_cat_search.best_estimator_, X_train, y_train, X_test, y_test)
evaluate_model("log_XGB_tuned",      log_xgb_search.best_estimator_, X_train, y_train, X_test, y_test)
evaluate_model("log_LGB_tuned",      log_lgb_search.best_estimator_, X_train, y_train, X_test, y_test)

In [ ]:
show_results()

In [ ]:
mlflow.end_run()